# 07 - Freddie Mac Feature Engineering and Leakage Review

## Milestone objective

Convert the validated 2015-2023 loan-level modeling dataset into a stable, auditable predictor dataset for Logistic Regression and XGBoost. This notebook explains why each source field is selected, transformed, retained only as metadata, or excluded.

The governing rule is simple: **a predictor must be known at origination and must be representable consistently when a future cohort arrives.**

## How features are chosen

Feature selection is not based on p-values alone. A p-value answers a narrow conditional-association question and can be misleading with hundreds of thousands of observations, nonlinear effects, correlated predictors, or temporal drift. We therefore use five gates:

1. **Timing:** Was the value known at loan origination?
2. **Domain relevance:** Is there a defensible relationship to borrower, collateral, or loan-contract risk?
3. **Temporal availability:** Is the field available throughout the 2015-2017 development window and usable later?
4. **Stability and transferability:** Will the field generalize across years and, later, to Fannie Mae?
5. **Redundancy and leakage:** Does it duplicate another field, act like an identifier, or contain post-origination information?

Statistical diagnostics will still be used later to understand relationships and model behavior, but they will not replace these design checks.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.freddie_features import (
    CATEGORICAL_FEATURES,
    LABEL_COLUMNS,
    NUMERIC_FEATURES,
    PREDICTOR_COLUMNS,
    apply_preprocessing_contract,
    build_engineered_features,
    feature_decision_table,
    fit_preprocessing_contract,
    validate_feature_dataset,
)

## Step 1 - Load the validated modeling dataset

Notebook 06 established one eligible row per loan, reconciled origination records to the 24-month target, and retained the event-window fields for auditing. We begin from that validated artifact rather than rebuilding labels here.

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'Freddie_Mac'
INPUT_PATH = DATA_DIR / 'freddie_modeling_dataset_2015_2023_24m.parquet'

modeling_data = pd.read_parquet(INPUT_PATH)
print('Input path:', INPUT_PATH)
print('Shape:', modeling_data.shape)
print('Years:', sorted(modeling_data['SOURCE YEAR'].unique().tolist()))
print('Duplicate loan/year keys:', modeling_data.duplicated(['SOURCE YEAR', 'LOAN IDENTIFIER']).sum())

## Step 2 - Review every origination field

The decision table is the audit trail for feature selection. It covers all 31 Freddie Mac origination fields. Exclusion does not mean a field is universally useless; it means the field is unsuitable for this project's stable, origination-time core model for the stated reason.

In [ ]:
decisions = feature_decision_table()
display(decisions)
display(decisions['decision'].value_counts().rename_axis('decision').to_frame('field_count'))

## Step 3 - Enforce the leakage boundary

Loan identifiers and cohort fields remain available for reconciliation, temporal splitting, and reporting, but they are not predictors. Event dates, observation-window fields, and the 24-month horizon are post-origination target-audit information and are removed entirely from the feature table. This prevents the model from receiving information that would not exist when a new loan is scored.

In [ ]:
engineered = build_engineered_features(modeling_data)

print('Engineered shape:', engineered.shape)
print('Predictors:', len(PREDICTOR_COLUMNS))
print('Numeric predictors:', len(NUMERIC_FEATURES))
print('Categorical predictors:', len(CATEGORICAL_FEATURES))
display(engineered.head())

## Step 4 - Create limited, domain-defensible features

The pipeline keeps engineering deliberately conservative:

- `LOG_ORIGINAL_UPB` reduces balance-scale skew without changing loan ordering.
- `CLTV_MINUS_LTV` measures the additional leverage attributable to subordinate financing. Raw CLTV is not also retained because LTV, CLTV, and their difference would be algebraically redundant for Logistic Regression.
- Mortgage-insurance, subordinate-financing, and LTV-above-80 indicators represent documented mortgage-domain thresholds or structures.
- Missingness flags preserve the possibility that non-disclosure itself carries information.

We do not create broad score or DTI bands at this stage. Keeping continuous information supports both models; bins can later be tested as an interpretable sensitivity analysis without replacing the core values.

In [ ]:
derived_columns = [
    'LOG_ORIGINAL_UPB',
    'CLTV_MINUS_LTV',
    'HAS_MORTGAGE_INSURANCE',
    'HAS_SUBORDINATE_FINANCING',
    'HIGH_LTV_GT_80',
]
display(engineered[derived_columns].describe().T)

missing_before_contract = (
    engineered[PREDICTOR_COLUMNS].isna().mean().mul(100).sort_values(ascending=False)
)
display(missing_before_contract[missing_before_contract.gt(0)].to_frame('missing_percent'))

## Step 5 - Fit preprocessing rules on 2015-2017 only

Medians and retained category levels are learned only from the development cohorts. Later cohorts must not influence preprocessing because that would allow future population information to leak backward into model development.

Numerical missing values receive development-period medians after explicit missingness indicators are created. Categorical missingness becomes `MISSING`; rare development categories and categories first observed in later years become `OTHER`. This gives all cohorts the same schema and makes previously unseen levels safe at scoring time. One-hot encoding is intentionally deferred to the model pipeline, where it will also be fitted on development data only.

In [ ]:
contract = fit_preprocessing_contract(
    engineered,
    minimum_category_count=100,
)
features = apply_preprocessing_contract(engineered, contract)

print('Contract training years:', contract['training_years'])
print('Minimum retained-category count:', contract['minimum_category_count'])
display(pd.Series(contract['numeric_medians'], name='training_median').to_frame())
display(pd.Series({key: len(value) for key, value in contract['category_levels'].items()}, name='retained_levels').to_frame())
print('Missing predictor cells after contract:', int(features[PREDICTOR_COLUMNS].isna().sum().sum()))

## Step 6 - Preserve the temporal experiment

The feature table assigns 2015-2017 to model development and 2018-2023 to sequential out-of-time evaluation. `SOURCE YEAR`, cohort, and role remain metadata rather than predictors. This is essential: the research question concerns whether a frozen model remains reliable as populations change, so the model must not be taught the answer by receiving calendar year as an input.

The target distribution is summarized but not resampled here. SMOTE, undersampling, or class weights would change the probability distribution and can damage calibration. Any later imbalance treatment must be evaluated inside the modeling design, not during dataset construction.

In [ ]:
role_summary = (
    features.groupby(['SOURCE YEAR', 'COHORT ROLE'], as_index=False)
    .agg(
        loans=('LOAN IDENTIFIER', 'size'),
        events=('TARGET_COMPOSITE_CREDIT_EVENT', 'sum'),
        event_rate=('TARGET_COMPOSITE_CREDIT_EVENT', 'mean'),
    )
)
role_summary['event_rate_percent'] = role_summary['event_rate'] * 100
display(role_summary.round(3))

## Step 7 - Validate and save the feature contract

The final checks require unique and complete loan/year keys, valid binary labels, no missing or infinite predictor values, both temporal roles, and complete exclusion of post-origination audit fields. The saved Parquet file is read back and validated again so serialization problems cannot pass silently.

In [ ]:
validation = validate_feature_dataset(features)
for key, value in validation.items():
    print(f'{key}: {value}')

OUTPUT_PATHS = {
    'features': DATA_DIR / 'freddie_features_2015_2023_24m.parquet',
    'decisions': DATA_DIR / 'freddie_feature_decisions.csv',
    'contract': DATA_DIR / 'freddie_feature_contract.json',
    'validation': DATA_DIR / 'freddie_feature_validation.json',
}

features.to_parquet(OUTPUT_PATHS['features'], index=False, engine='pyarrow')
decisions.to_csv(OUTPUT_PATHS['decisions'], index=False)
with OUTPUT_PATHS['contract'].open('w', encoding='utf-8') as file:
    json.dump(contract, file, indent=2)
with OUTPUT_PATHS['validation'].open('w', encoding='utf-8') as file:
    json.dump(validation, file, indent=2)

saved_features = pd.read_parquet(OUTPUT_PATHS['features'])
saved_validation = validate_feature_dataset(saved_features)
assert validation == saved_validation

print('Saved and read-back validated:')
for name, path in OUTPUT_PATHS.items():
    print(f'  {name}: {path}')

## Milestone conclusion

The project now has a documented, reproducible feature boundary rather than an informal list of columns. All predictors are available at origination; post-origination target evidence is excluded; definition changes are either harmonized or rejected; preprocessing is frozen from 2015-2017; and the same feature contract can be applied to every 2018-2023 cohort.

The next research milestone is the peer-reviewed literature review on population drift, performance degradation, early-warning signals, and retraining benefit. After the benchmark methodology is grounded in that literature, the same frozen feature table will support Logistic Regression and XGBoost training.